# Reviewer 3: sinusoidal positional encoding / attention periodicity

Reviewer 3 noted a periodicity of ~15 genes in the attention heatmap (Fig 4D) and
suspected it is an artifact of the sinusoidal positional encoding interacting with the
circular relative attention mechanism, rather than a true biological signal. They asked
for three things:

1. Attention patterns from a model **without sinusoidal encodings** (learnable circular
   relative attention only).
2. Attention patterns from an **untrained/randomly initialised** model, to see whether the
   periodicity exists before any learning has occurred.
3. Whether genes at the periodic distances **actually show functional relationships** - if
   they don't, that supports the pattern being artifactual.

This notebook compares all three model types (`original`, `no_sinusoidal`, `untrained`),
each trained/evaluated across the same 10 folds on Setonix. It does **not** load any model
checkpoints or raw training output - only the small summary files produced by
`scripts/analyze_periodicity.py`, which aggregates the out-of-fold (OOF) validation
attention that was already saved during training (`final_validation_attention.pkl` etc.,
per fold). That's what makes this tractable locally: the full trained_models directory is
~3TB, but these summaries are a few KB each.

**Before running this notebook:** copy the three summary files down from Setonix, e.g.
```bash
scp grig0076@setonix.pawsey.org.au:/home/grig0076/susie_scratch/phynteny_transformer/periodicity_results/periodicity_*.npz .
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MODEL_TYPES = ["original", "no_sinusoidal", "untrained"]
COLORS = {"original": "#332288", "no_sinusoidal": "#117733", "untrained": "#999933"}

summaries = {}
for model_type in MODEL_TYPES:
    path = f"periodicity_{model_type}.npz"
    summaries[model_type] = np.load(path)
    print(model_type, "loaded:", list(summaries[model_type].keys()))


## 1. Attention as a function of relative gene distance

This is the 1D analogue of the off-diagonal periodicity in the Fig 4D heatmap: mean
(head-averaged) attention value at each signed relative distance between a query and key
gene position, aggregated across every validation genome and fold for that model type.
If the ~15-gene periodicity is a real, consistent property of the model's attention, it
should show up as regularly spaced peaks in this profile - and if it's driven by the
sinusoidal encoding, it should weaken or disappear once that encoding is removed
(`no_sinusoidal`) or never trained at all (`untrained`).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, model_type in zip(axes, MODEL_TYPES):
    d = summaries[model_type]
    offsets = d["offsets"]
    profile = d["attn_profile"]
    ax.plot(offsets, profile, color=COLORS[model_type])
    ax.axvline(0, color="grey", linewidth=0.5)
    for k in (-30, -15, 15, 30):
        ax.axvline(k, color="grey", linewidth=0.5, linestyle=":")
    ax.set_title(model_type)
    ax.set_xlabel("Relative gene distance (key - query)")

axes[0].set_ylabel("Mean attention")
plt.tight_layout()
plt.savefig("periodicity_comparison_profiles.png", dpi=300, bbox_inches="tight")
plt.show()


## 2. Periodicity strength (autocorrelation)

`scripts/analyze_periodicity.py` already computes the autocorrelation of each profile
over a range of lags and reports the lag with maximum autocorrelation
(`best_period`/`best_autocorr`). Comparing this number across the three model types is
the direct, quantitative answer to Reviewer 3's core question.


In [ ]:
summary_rows = []
for model_type in MODEL_TYPES:
    d = summaries[model_type]
    best_period = int(d["periodicity_best_period"])
    best_autocorr = float(d["periodicity_best_autocorr"])
    summary_rows.append((model_type, best_period, best_autocorr))
    print(f"{model_type:15s}  best_period={best_period:3d}  best_autocorr={best_autocorr:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
for model_type in MODEL_TYPES:
    d = summaries[model_type]
    ax.plot(d["periodicity_lags"], d["periodicity_autocorr"], label=model_type, color=COLORS[model_type])
ax.axvline(15, color="grey", linewidth=0.5, linestyle=":")
ax.set_xlabel("Lag (genes)")
ax.set_ylabel("Autocorrelation")
ax.legend()
plt.tight_layout()
plt.savefig("periodicity_autocorrelation.png", dpi=300, bbox_inches="tight")
plt.show()


## 3. Do genes at the periodic distance actually show functional relationships?

For each model type, this is the fraction of gene pairs sharing the same PHROG category,
as a function of relative distance - restricted to real gene pairs where both genes have
a known category. If the periodic distance (e.g. ±15 genes) doesn't show elevated
functional agreement compared to nearby non-periodic distances, that supports Reviewer 3's
suspicion that the attention periodicity is architectural rather than biological.

Note this profile is computed once from the data (not per model type in a meaningful way,
since it only depends on gene positions/categories, not on the model) - but it's saved per
run for convenience; they should be near-identical across model types since they use the
same validation folds.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
d = summaries["original"]
ax.plot(d["cat_offsets"], d["agreement_profile"], color="black")
ax.axvline(0, color="grey", linewidth=0.5)
for k in (-30, -15, 15, 30):
    ax.axvline(k, color="grey", linewidth=0.5, linestyle=":")
ax.set_xlabel("Relative gene distance")
ax.set_ylabel("P(same PHROG category)")
plt.tight_layout()
plt.savefig("periodicity_functional_agreement.png", dpi=300, bbox_inches="tight")
plt.show()

# Quantify: agreement at the periodic distance vs a matched non-periodic baseline
offsets = d["cat_offsets"]
agreement = d["agreement_profile"]
periodic_mask = np.isin(offsets, [-15, 15])
baseline_mask = np.isin(offsets, [-10, -20, 10, 20])  # nearby, non-periodic distances
print("Mean agreement at +-15 (periodic):", np.nanmean(agreement[periodic_mask]))
print("Mean agreement at nearby non-periodic distances:", np.nanmean(agreement[baseline_mask]))


## Summary for the rebuttal

Fill in once the above has been run against the real Setonix results:

- **Without sinusoidal encoding:** [does the ~15-gene periodicity persist, weaken, or
  disappear in `no_sinusoidal` relative to `original`?]
- **Untrained model:** [is the periodicity present before any learning, i.e. is it purely
  a property of the circular relative attention's initialisation/architecture?]
- **Functional relationships:** [is there elevated PHROG-category agreement specifically
  at the periodic distance, or is it flat/no different from nearby distances?]

Together these three results directly answer Reviewer 3's request: whether the observed
periodicity reflects learned biological structure or an architectural artifact of the
positional encoding and/or circular relative attention.
